# Create FWO (Research Foundation - Flanders) Awards

**FWO / Fonds Wetenschappelijk Onderzoek - Vlaanderen** (Research Foundation - Flanders) is the main funder of fundamental scientific research in Flanders, Belgium. This ingest covers FWO-funded research projects and personal fellowships/mandates.

**Source — FRIS (Flanders Research Information Space), Method 2 (SOAP/XML API).** `scripts/local/fwo_to_s3.py` harvests the public `ProjectServiceFRIS.getProjects` web service (`frisr4.researchportal.be`), which holds **all** Flemish research projects across many funders, and keeps only those whose **Funding Party** is "Research Foundation Flanders" (= FWO). NOTE: the string "FWO" also appears on non-FWO projects via the `fwoDisciplines` classification — membership is decided by the funding organisation, never by a substring.

**Awarding body funder:** Research Foundation - Flanders — funder_id **4320321730** (BE; `F4320*` ⇒ Path A, present in `openalex.common.funder`).

**Schema choices:**
- `funder_award_id` = the **real FWO contract id** (`fundingIdentifier` @authority=`FWO Contract Id`, e.g. `1286227N`) — the grant reference cited in paper acknowledgements (real work-linkage). Older projects without one fall back to `fwo-{project_uuid}`.
- `display_name` = the project title (EN, fallback NL). `description` = the EN abstract.
- `funding_type` = `fellowship` for personal mandates/fellowships, else `grant`. `funder_scheme` = the FWO funding scheme (e.g. "FWO junior postdoctoral fellowship").
- `start_date`/`end_date` = real project dates (day precision); `start_year`/`end_year` derived (future-year cap applied).
- `lead_investigator` = the **promoter** (given/family explicit from FRIS — no name-splitting); `co_lead_investigator` = the co-promoter. `affiliation.name` = the host university, `country` = BE.
- **amount/currency NULL (§6.7 waiver)** — FRIS does not publish a per-project budget for FWO grants (`nullBudgetAllowed=true`); never imputed.

**S3:** `s3a://openalex-ingest/awards/fwo/fwo_projects.parquet`  ·  **provenance:** `fwo_fris`


## Step 1: Staging table from S3


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.fwo_raw
USING delta AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/fwo/fwo_projects.parquet`;


In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.fwo_raw;


## Step 1.5: Inspect raw + coverage

Expected: ~60-65k FWO project rows, ~2010-2029. title / lead_family_name / institution / start_date / funder_scheme ~100%; real FWO grant id high.


In [ ]:
%sql
DESCRIBE openalex.awards.fwo_raw;


In [ ]:
%sql
SELECT funder_award_id, fwo_grant_id, lead_given_name, lead_family_name, institution_name,
       start_date, end_date, funding_type, funder_scheme, SUBSTRING(title,1,60) AS title
FROM openalex.awards.fwo_raw LIMIT 8;


In [ ]:
%sql
SELECT funder_scheme, funding_type, COUNT(*) AS n
FROM openalex.awards.fwo_raw
GROUP BY funder_scheme, funding_type ORDER BY n DESC LIMIT 15;


In [ ]:
%sql
SELECT COUNT(*) AS total,
       COUNT(fwo_grant_id) AS has_real_grant_id,
       COUNT(DISTINCT funder_award_id) AS distinct_ids,
       COUNT(title) AS has_title, COUNT(lead_family_name) AS has_lead,
       COUNT(institution_name) AS has_inst, COUNT(start_date) AS has_start
FROM openalex.awards.fwo_raw;


## Step 1.6: Fail-fast — verify FWO funder row exists (Path A)

`4320321730` is `F4320*`, so it MUST be present in `openalex.common.funder`. This query must return exactly 1 row; if 0, STOP (the Step-2 CROSS JOIN would silently emit 0 rows).


In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320321730;


## Step 2: Transform to award schema

GRANT pattern with a named promoter. amount/currency NULL (§6.7 FWO budget waiver). Future-year cap (§2.3): `start_year > YEAR(current_date())+1` ⇒ NULL both years.


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.fwo_awards
USING delta AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320321730   -- Research Foundation - Flanders (FWO)
),
base AS (
    SELECT
        n.funder_award_id,
        n.title, n.abstract, n.funder_scheme, n.funding_type,
        n.lead_given_name, n.lead_family_name,
        n.colead_given_name, n.colead_family_name,
        n.institution_name, n.country, n.landing_page_url,
        TRY_TO_DATE(n.start_date, 'yyyy-MM-dd') AS sd,
        TRY_TO_DATE(n.end_date,   'yyyy-MM-dd') AS ed,
        f.funder_id, f.display_name AS f_name, f.ror_id, f.doi AS f_doi
    FROM openalex.awards.fwo_raw n
    CROSS JOIN funder_resolved f
    WHERE n.funder_award_id IS NOT NULL
)
SELECT
    abs(xxhash64(CONCAT(TRY_CAST(funder_id AS STRING), ':', LOWER(funder_award_id)))) % 9000000000 AS id,
    title AS display_name,
    abstract AS description,
    funder_id,
    funder_award_id,
    CAST(NULL AS DOUBLE) AS amount,
    CAST(NULL AS STRING) AS currency,
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(funder_id AS STRING)) AS id,
        f_name AS display_name, ror_id AS ror_id, f_doi AS doi
    ) AS funder,
    funding_type,
    funder_scheme,
    'fwo_fris' AS provenance,
    sd AS start_date,
    ed AS end_date,
    CASE WHEN YEAR(sd) > YEAR(current_date()) + 1 THEN NULL ELSE YEAR(sd) END AS start_year,
    CASE WHEN YEAR(sd) > YEAR(current_date()) + 1 THEN NULL ELSE YEAR(ed) END AS end_year,
    CASE WHEN lead_given_name IS NOT NULL OR lead_family_name IS NOT NULL THEN struct(
        lead_given_name AS given_name, lead_family_name AS family_name,
        CAST(NULL AS STRING) AS orcid, CAST(NULL AS DATE) AS role_start,
        struct(institution_name AS name, country AS country,
               CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids) AS affiliation
    ) ELSE NULL END AS lead_investigator,
    CASE WHEN colead_given_name IS NOT NULL OR colead_family_name IS NOT NULL THEN struct(
        colead_given_name AS given_name, colead_family_name AS family_name,
        CAST(NULL AS STRING) AS orcid, CAST(NULL AS DATE) AS role_start,
        struct(institution_name AS name, country AS country,
               CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids) AS affiliation
    ) ELSE NULL END AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    landing_page_url,
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT(TRY_CAST(funder_id AS STRING), ':', LOWER(funder_award_id)))) % 9000000000 AS STRING)) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM base;


## Step 3: Insert into openalex_awards_raw at priority 167


In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'fwo_fris' AND priority = 167;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    167 AS priority  -- FWO (Research Foundation - Flanders)
FROM openalex.awards.fwo_awards;


## Step 6: Verification

§6.7 amount waiver **applies** (FRIS publishes no per-project FWO budget; amount/currency NULL by design). Completeness: display_name / lead_investigator / start_year ~100%.


In [ ]:
%sql
SELECT COUNT(*) AS total_rows FROM openalex.awards.fwo_awards;


In [ ]:
%sql
DESCRIBE openalex.awards.fwo_awards;


In [ ]:
%sql
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(lead_investigator) AS has_lead,
    COUNT(start_year) AS has_start_year,
    COUNT(funder_scheme) AS has_scheme,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) AS pct_title,
    ROUND(try_divide(COUNT(lead_investigator), COUNT(*)) * 100.0, 1) AS pct_lead,
    ROUND(try_divide(COUNT(start_year), COUNT(*)) * 100.0, 1) AS pct_start_year
FROM openalex.awards.fwo_awards;


In [ ]:
%sql
-- §6.4a sanity: no single promoter or title should dominate (catches scraper DOM-node bugs)
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.fwo_awards
GROUP BY lead_investigator.given_name, lead_investigator.family_name
ORDER BY n DESC LIMIT 10;


In [ ]:
%sql
SELECT MIN(start_year) AS first_year, MAX(start_year) AS last_year,
       COUNT(*) AS total, COUNT(DISTINCT id) AS distinct_ids,
       SUM(CASE WHEN start_year > YEAR(current_date())+1 THEN 1 ELSE 0 END) AS future_year_rows
FROM openalex.awards.fwo_awards;


In [ ]:
%sql
SELECT funder.id, funder.display_name, funder.ror_id, funder.doi, COUNT(*) AS rows
FROM openalex.awards.fwo_awards
GROUP BY funder.id, funder.display_name, funder.ror_id, funder.doi;


In [ ]:
%sql
SELECT id, SUBSTRING(display_name, 1, 50) AS title, funding_type, start_year,
       lead_investigator.given_name AS given, lead_investigator.family_name AS family,
       lead_investigator.affiliation.name AS institution
FROM openalex.awards.fwo_awards
ORDER BY start_year DESC LIMIT 10;
